# BookWise — E-Commerce Book Pricing & Rating Analytics

## 02. Data Cleaning & Preprocessing

### Objective

The objective of this notebook is to transform the raw enriched web-scraped dataset into a clean, consistent, analysis-ready dataset.


In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_product_url = "https://books.toscrape.com/catalogue/"

### Load the Enriched Raw Dataset

In [ ]:
file_path = "data/enriched_raw_books.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (1000, 12)


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Title              1000 non-null   str  
 1   Price              1000 non-null   str  
 2   Rating             1000 non-null   str  
 3   Availability       1000 non-null   str  
 4   Link               1000 non-null   str  
 5   Category           1000 non-null   str  
 6   Description        998 non-null    str  
 7   UPC                1000 non-null   str  
 8   Product Type       1000 non-null   str  
 9   Tax                1000 non-null   str  
 10  Number Available   1000 non-null   str  
 11  Number of Reviews  1000 non-null   int64
dtypes: int64(1), str(11)
memory usage: 1.6 MB


In [ ]:
df.head()

,Title,Price,Rating,Availability,Link,Category,Description,UPC,Product Type,Tax,Number Available,Number of Reviews
0,A Light in the Attic,Â£51.77,Three,In stock,a-light-in-the-attic_1000/index.html,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,Books,Â£0.00,In stock (22 available),0
1,Tipping the Velvet,Â£53.74,One,In stock,tipping-the-velvet_999/index.html,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,Books,Â£0.00,In stock (20 available),0
2,Soumission,Â£50.10,One,In stock,soumission_998/index.html,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760,Books,Â£0.00,In stock (20 available),0
3,Sharp Objects,Â£47.82,Four,In stock,sharp-objects_997/index.html,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,Books,Â£0.00,In stock (20 available),0
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,sapiens-a-brief-history-of-humankind_996/index...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,Books,Â£0.00,In stock (20 available),0


In [ ]:
df.tail()

,Title,Price,Rating,Availability,Link,Category,Description,UPC,Product Type,Tax,Number Available,Number of Reviews
995,Alice in Wonderland (Alice's Adventures in Won...,Â£55.53,One,In stock,alice-in-wonderland-alices-adventures-in-wonde...,Classics,NaN,cd2a2a70dd5d176d,Books,Â£0.00,In stock (1 available),0
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Â£57.06,Four,In stock,ajin-demi-human-volume-1-ajin-demi-human-1_4/i...,Sequential Art,High school student Kei Nagai is struck dead i...,bfd5e1701c862ac3,Books,Â£0.00,In stock (1 available),0
997,A Spy's Devotion (The Regency Spies of London #1),Â£16.97,Five,In stock,a-spys-devotion-the-regency-spies-of-london-1_...,Historical Fiction,"In Englandâs Regency era, manners and elegan...",19fec36a1dfb4c16,Books,Â£0.00,In stock (1 available),0
998,1st to Die (Women's Murder Club #1),Â£53.98,One,In stock,1st-to-die-womens-murder-club-1_2/index.html,Mystery,"James Patterson, bestselling author of the Ale...",f684a82adc49f011,Books,Â£0.00,In stock (1 available),0
999,"1,000 Places to See Before You Die",Â£26.08,Five,In stock,1000-places-to-see-before-you-die_1/index.html,Travel,"Around the World, continent by continent, here...",228ba5e7577e1d49,Books,Â£0.00,In stock (1 available),0


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 1000
Number of columns: 12


### Missing Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary

,Missing Values,Missing Percentage
Title,0,0.0
Price,0,0.0
Rating,0,0.0
Availability,0,0.0
Link,0,0.0
Category,0,0.0
Description,2,0.2
UPC,0,0.0
Product Type,0,0.0
Tax,0,0.0


### Duplicate Record Analysis

In [ ]:
duplicate_count = df.duplicated().sum()

print("Complete duplicate records:", duplicate_count)

Complete duplicate records: 0


### Text Encoding Correction

Some text values contain character-encoding artifacts such as "Â£" and "Ã".

These artifacts can affect price conversion, text analysis, NLP, searching, and visualization.

We will correct the affected text fields before continuing with further preprocessing.

In [ ]:
def fix_encoding(text):
    if pd.isna(text):
        return text
    
    try:
        return text.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text

In [ ]:
text_columns = [
    "Title",
    "Price",
    "Rating",
    "Availability",
    "Link",
    "Category",
    "Description",
    "UPC",
    "Product Type",
    "Tax",
    "Number Available"
]

for column in text_columns:
    df[column] = df[column].apply(fix_encoding)

In [ ]:
print(df["Price"].head())
print()
print(df["Description"].head(3))

0    £51.77
1    £53.74
2    £50.10
3    £47.82
4    £54.23
Name: Price, dtype: str

0    It's hard to imagine a world without A Light i...
1    "Erotic and absorbing...Written with starling ...
2    Dans une France assez proche de la nôtre, un h...
Name: Description, dtype: str


### Convert Price and Tax to Numeric Values

Price and tax were extracted from the website as text values containing the pound (£) symbol.

I removed the currency symbol and convert these fields into numeric variables so they can be used in calculations and statistical analysis.

In [ ]:
df["Price"] = (
    df["Price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

df["Tax"] = (
    df["Tax"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

print(df[["Price", "Tax"]].head())

   Price  Tax
0  51.77  0.0
1  53.74  0.0
2  50.10  0.0
3  47.82  0.0
4  54.23  0.0


### Convert Book Ratings to Numeric Values

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["Rating"] = df["Rating"].map(rating_map)

print(df["Rating"].value_counts().sort_index())

Rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64


### Extract Available Stock Quantity

In [ ]:
df["Stock_Quantity"] = (
    df["Number Available"]
    .str.extract(r"(\d+)")
    .astype(int)
)

print(df[["Number Available", "Stock_Quantity"]].head())

          Number Available  Stock_Quantity
0  In stock (22 available)              22
1  In stock (20 available)              20
2  In stock (20 available)              20
3  In stock (20 available)              20
4  In stock (20 available)              20


### Create Rating Categories

In [ ]:
def rating_category(rating):
    if rating <= 2:
        return "Low"
    elif rating == 3:
        return "Average"
    else:
        return "High"

df["Rating_Category"] = df["Rating"].apply(rating_category)

print(df["Rating_Category"].value_counts())

Rating_Category
Low        422
High       375
Average    203
Name: count, dtype: int64


In [ ]:
df[[
    "Title",
    "Price",
    "Tax",
    "Rating",
    "Stock_Quantity",
    "Rating_Category"
]].head(10)

,Title,Price,Tax,Rating,Stock_Quantity,Rating_Category
0,A Light in the Attic,51.77,0.0,3,22,Average
1,Tipping the Velvet,53.74,0.0,1,20,Low
2,Soumission,50.10,0.0,1,20,Low
3,Sharp Objects,47.82,0.0,4,20,High
4,Sapiens: A Brief History of Humankind,54.23,0.0,5,20,High
5,The Requiem Red,22.65,0.0,1,19,Low
6,The Dirty Little Secrets of Getting Your Dream...,33.34,0.0,4,19,High
7,The Coming Woman: A Novel Based on the Life of...,17.93,0.0,3,19,Average
8,The Boys in the Boat: Nine Americans and Their...,22.60,0.0,4,19,High
9,The Black Maria,52.15,0.0,1,19,Low


### Handle Missing Product Descriptions

In [ ]:
print("Missing descriptions:", df["Description"].isna().sum())

print("\nRows with missing descriptions:")
df[df["Description"].isna()][["Title", "Description"]]

Missing descriptions: 2

Rows with missing descriptions:


,Title,Description
160,The Bridge to Consciousness: I'm Writing the B...,NaN
995,Alice in Wonderland (Alice's Adventures in Won...,NaN


In [ ]:
missing_before = df["Description"].isnull().sum()

df["Description"] = df["Description"].fillna("Description not available")

missing_after = df["Description"].isnull().sum()

print("Missing descriptions before:", missing_before)
print("Missing descriptions after:", missing_after)

Missing descriptions before: 2
Missing descriptions after: 0


### Text Cleaning

In [ ]:
import re

### Standardize Text Fields

Text fields may contain unnecessary spaces, inconsistent capitalization, and unwanted special characters.

We will standardize the text while preserving meaningful characters.

In [ ]:
df["Title"] = (
    df["Title"]
    .astype(str)
    .str.strip()
)

df["Description"] = (
    df["Description"]
    .astype(str)
    .str.strip()
)

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
)

print(df[["Title", "Category"]].head(10))

                                               Title            Category
0                               A Light in the Attic              Poetry
1                                 Tipping the Velvet  Historical Fiction
2                                         Soumission             Fiction
3                                      Sharp Objects             Mystery
4              Sapiens: A Brief History of Humankind             History
5                                    The Requiem Red         Young Adult
6  The Dirty Little Secrets of Getting Your Dream...            Business
7  The Coming Woman: A Novel Based on the Life of...             Default
8  The Boys in the Boat: Nine Americans and Their...             Default
9                                    The Black Maria              Poetry


### Create Text-Based Features

Text length and word count can provide useful information about how detailed a book's title and description are.

In [ ]:
df["Title_Length"] = df["Title"].str.len()

df["Description_Length"] = df["Description"].str.len()

df["Description_Word_Count"] = (
    df["Description"]
    .str.split()
    .str.len()
)

df[[
    "Title",
    "Title_Length",
    "Description_Length",
    "Description_Word_Count"
]].head(10)

,Title,Title_Length,Description_Length,Description_Word_Count
0,A Light in the Attic,20,1017,164
1,Tipping the Velvet,18,1029,165
2,Soumission,10,1093,163
3,Sharp Objects,13,1635,274
4,Sapiens: A Brief History of Humankind,37,1945,304
5,The Requiem Red,15,995,170
6,The Dirty Little Secrets of Getting Your Dream...,50,1857,284
7,The Coming Woman: A Novel Based on the Life of...,87,3411,568
8,The Boys in the Boat: Nine Americans and Their...,94,2042,335
9,The Black Maria,15,2979,464


### Create Price Bands

To make pricing patterns easier to interpret, books will be grouped into meaningful price bands.

These categories can be used to compare ratings, stock levels, categories, and review activity across different price ranges.

In [ ]:
def price_band(price):
    if price < 20:
        return "Budget"
    elif price < 40:
        return "Mid-Range"
    elif price < 60:
        return "Premium"
    else:
        return "Luxury"

df["Price_Band"] = df["Price"].apply(price_band)

print(df["Price_Band"].value_counts())

Price_Band
Premium      403
Mid-Range    401
Budget       196
Name: count, dtype: int64


### Create Inventory Status

Classifying products into Low, Medium, and High stock levels.

In [ ]:
df["Stock_Status"] = pd.cut(
    df["Stock_Quantity"],
    bins=[0, 10, 15, float("inf")],
    labels=["Low Stock", "Medium Stock", "High Stock"],
    include_lowest=True
)

print(df["Stock_Status"].value_counts())

NameError: name 'pd' is not defined

### Create Review Activity Categories

The number of reviews can be used as an indicator of customer engagement.

In [ ]:
def review_activity(reviews):
    if reviews == 0:
        return "No Reviews"
    elif reviews <= 5:
        return "Low"
    elif reviews <= 20:
        return "Medium"
    else:
        return "High"

df["Review_Activity"] = df["Number of Reviews"].apply(review_activity)

print(df["Review_Activity"].value_counts())

Review_Activity
No Reviews    1000
Name: count, dtype: int64


In [ ]:
df["Price_Band"].value_counts()

Price_Band
Premium      403
Mid-Range    401
Budget       196
Name: count, dtype: int64

In [ ]:
df["Review_Activity"].value_counts()

Review_Activity
No Reviews    1000
Name: count, dtype: int64

### Remove Non-Informative Review Activity Feature

The source dataset contains zero reviews for all 1,000 books.

Since the `Number of Reviews` variable has no variation, a review-activity classification would not provide useful analytical information.

Therefore, the derived `Review_Activity` feature will not be retained.

In [ ]:
df.drop(columns=["Review_Activity"], inplace=True)

print("Review_Activity feature removed.")

Review_Activity feature removed.


### Analyze Categorical Variables

Categorical variables such as Category, Product Type, Price Band, Stock Status, and Rating Category will be inspected for consistency and unique values.

This ensures that categories are standardized before they are used.

In [ ]:
print("Number of Categories:", df["Category"].nunique())
print("\nTop Categories:")
print(df["Category"].value_counts().head(15))

Number of Categories: 50

Top Categories:
Category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               19
History                18
Name: count, dtype: int64


In [ ]:
print("Product Types:")
print(df["Product Type"].value_counts())

Product Types:
Product Type
Books    1000
Name: count, dtype: int64


In [ ]:
print("Rating Categories:")
print(df["Rating_Category"].value_counts())

Rating Categories:
Rating_Category
Low        422
High       375
Average    203
Name: count, dtype: int64


In [ ]:
print("Price Bands:")
print(df["Price_Band"].value_counts())

Price Bands:
Price_Band
Premium      403
Mid-Range    401
Budget       196
Name: count, dtype: int64


In [ ]:
print("Stock Status:")
print(df["Stock_Status"].value_counts())

Stock Status:
Stock_Status
Low Stock       582
Medium Stock    290
High Stock      128
Name: count, dtype: int64


### Handle Unclassified Source Categories

During data validation, 67 records contain "Add a comment" as their category.

Inspection of the original product page confirmed that this value is provided by the source website itself rather than being introduced during our scraping process.

Since the actual category cannot be reliably determined from the available source data, these records will be retained without manually assigning a category.

In [ ]:
unknown_count = (df["Category"] == "Add a comment").sum()

df["Category"] = df["Category"].replace(
    "Add a comment",
    "Unknown"
)

print("Categories standardized:", unknown_count)
print("Unknown categories:", (df["Category"] == "Unknown").sum())
print("Total records retained:", len(df))

Categories standardized: 67
Unknown categories: 67
Total records retained: 1000


### Final Data Quality Validation

The cleaned dataset will now be validated to ensure that the original records have been retained, missing values have been handled, duplicate records are absent, and the engineered features are available for downstream analysis.

In [ ]:
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))
print("Duplicate Records:", df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

Total Records: 1000
Total Columns: 19
Duplicate Records: 0

Missing Values:
Title                     0
Price                     0
Rating                    0
Availability              0
Link                      0
Category                  0
Description               0
UPC                       0
Product Type              0
Tax                       0
Number Available          0
Number of Reviews         0
Stock_Quantity            0
Rating_Category           0
Title_Length              0
Description_Length        0
Description_Word_Count    0
Price_Band                0
Stock_Status              0
dtype: int64


In [ ]:
print("Data Types:")
print(df.dtypes)

Data Types:
Title                          str
Price                      float64
Rating                       int64
Availability                   str
Link                           str
Category                       str
Description                    str
UPC                            str
Product Type                   str
Tax                        float64
Number Available               str
Number of Reviews            int64
Stock_Quantity               int64
Rating_Category                str
Title_Length                 int64
Description_Length           int64
Description_Word_Count       int64
Price_Band                     str
Stock_Status              category
dtype: object


In [ ]:
df[
    [
        "Title",
        "Price",
        "Rating",
        "Category",
        "Stock_Quantity",
        "Rating_Category",
        "Title_Length",
        "Description_Length",
        "Description_Word_Count",
        "Price_Band",
        "Stock_Status"
    ]
].head(10)

,Title,Price,Rating,Category,Stock_Quantity,Rating_Category,Title_Length,Description_Length,Description_Word_Count,Price_Band,Stock_Status
0,A Light in the Attic,51.77,3,Poetry,22,Average,20,1017,164,Premium,High Stock
1,Tipping the Velvet,53.74,1,Historical Fiction,20,Low,18,1029,165,Premium,High Stock
2,Soumission,50.10,1,Fiction,20,Low,10,1093,163,Premium,High Stock
3,Sharp Objects,47.82,4,Mystery,20,High,13,1635,274,Premium,High Stock
4,Sapiens: A Brief History of Humankind,54.23,5,History,20,High,37,1945,304,Premium,High Stock
5,The Requiem Red,22.65,1,Young Adult,19,Low,15,995,170,Mid-Range,High Stock
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,Business,19,High,50,1857,284,Mid-Range,High Stock
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,Default,19,Average,87,3411,568,Budget,High Stock
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,Default,19,High,94,2042,335,Mid-Range,High Stock
9,The Black Maria,52.15,1,Poetry,19,Low,15,2979,464,Premium,High Stock


In [ ]:
output_path = "data/cleaned_books.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully.")
print("File:", output_path)
print("Records:", len(df))
print("Columns:", len(df.columns))

Cleaned dataset saved successfully.
File: data/cleaned_books.csv
Records: 1000
Columns: 19


In [ ]:
cleaned_df = pd.read_csv(
    "data/cleaned_books.csv",
    encoding="utf-8-sig"
)

print("Final verification")
print("------------------")
print("Records:", cleaned_df.shape[0])
print("Columns:", cleaned_df.shape[1])
print("Duplicates:", cleaned_df.duplicated().sum())
print("Missing values:", cleaned_df.isnull().sum().sum())

Final verification
------------------
Records: 1000
Columns: 19
Duplicates: 0
Missing values: 0


## Conclusion

The raw BookWise dataset was successfully cleaned, validated, and transformed into an analysis-ready dataset.

The final dataset contains 1,000 records and 19 features with no duplicate records and no missing values.

Data preprocessing included handling inconsistent formats, converting numerical fields into appropriate data types, extracting stock quantities, and creating additional analytical features such as Rating_Category, Price_Band, Stock_Status, Title_Length, Description_Length, and Description_Word_Count.

The resulting cleaned dataset was saved as a CSV file and used as the primary dataset for SQL analysis, exploratory data analysis, and machine learning.